# entrenamiento_prediccion_inundaciones

In [1]:
import os, json, time, hashlib, warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import xgboost as xgb
import lightgbm as lgb
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             f1_score, precision_recall_curve)

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

DIR_ML = Path(os.environ.get("DIR_ML", Path.cwd())).resolve()
DIR_CLIMA, DIR_INS, DIR_OUT = DIR_ML / "clima", DIR_ML / "insumos", DIR_ML / "salidas"
DIR_OUT.mkdir(parents=True, exist_ok=True)
SEMILLA = 42
print("Carpeta de trabajo:", DIR_ML)
for d in (DIR_CLIMA / "ideam_resume", DIR_CLIMA / "condiciones", DIR_INS / "sar40", DIR_INS / "clases", DIR_INS / "terreno"):
    print(f"  {'OK ' if d.exists() else 'FALTA'}  {d.relative_to(DIR_ML)}")

Carpeta de trabajo: /home/claude/proyecto/_ml_entrenamiento
  OK   clima/ideam_resume
  OK   clima/condiciones
  OK   insumos/sar40
  OK   insumos/clases
  OK   insumos/terreno


## Common functions: models and metrics

In [2]:
def crear_modelos(peso_pos):
    imp = lambda: SimpleImputer(strategy="median", add_indicator=True)
    return {
        "Regresión logística": make_pipeline(imp(), StandardScaler(),
                                             LogisticRegression(max_iter=3000, C=0.5, class_weight="balanced")),
        "Random Forest": make_pipeline(imp(), RandomForestClassifier(
            n_estimators=300, min_samples_leaf=5, max_features="sqrt", class_weight="balanced_subsample",
            n_jobs=-1, random_state=SEMILLA)),
        "HistGradientBoosting": HistGradientBoostingClassifier(
            max_iter=400, learning_rate=0.05, max_leaf_nodes=31, l2_regularization=1.0,
            class_weight="balanced", early_stopping=False, random_state=SEMILLA),
        "XGBoost": xgb.XGBClassifier(
            n_estimators=500, learning_rate=0.03, max_depth=5, subsample=0.8, colsample_bytree=0.8,
            min_child_weight=3, scale_pos_weight=peso_pos, eval_metric="aucpr", n_jobs=-1, random_state=SEMILLA),
        "LightGBM": lgb.LGBMClassifier(
            n_estimators=500, learning_rate=0.03, num_leaves=31, subsample=0.8, subsample_freq=1,
            colsample_bytree=0.8, min_child_samples=20, scale_pos_weight=peso_pos, random_state=SEMILLA, verbose=-1),
        "Red neuronal (MLP)": make_pipeline(imp(), StandardScaler(), MLPClassifier(
            hidden_layer_sizes=(64, 32), alpha=1e-3, learning_rate_init=1e-3, max_iter=400,
            early_stopping=True, n_iter_no_change=20, random_state=SEMILLA)),
    }


def umbral_max_f1(y, p):
    prec, rec, umb = precision_recall_curve(y, p)
    f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-12)
    i = int(np.nanargmax(f1[:-1])) if len(umb) else 0
    return float(umb[i]) if len(umb) else 0.5


def metricas(y, p, umbral):
    y = np.asarray(y).astype(int); p = np.asarray(p, float); yhat = p >= umbral
    tp = int((yhat & (y == 1)).sum()); fp = int((yhat & (y == 0)).sum()); fn = int((~yhat & (y == 1)).sum())
    return {
        "ROC_AUC": roc_auc_score(y, p) if 0 < y.sum() < len(y) else np.nan,
        "PR_AUC": average_precision_score(y, p) if y.sum() else np.nan,
        "F1": f1_score(y, yhat, zero_division=0),
        "POD": tp / (tp + fn) if tp + fn else np.nan,
        "FAR": fp / (tp + fp) if tp + fp else np.nan,
        "CSI": tp / (tp + fp + fn) if tp + fp + fn else np.nan,
        "Brier": brier_score_loss(y, np.clip(p, 0, 1)),
        "umbral": umbral,
    }

# Part A · Hydrological model: when does it flood?
## A1. IDEAM, GloFAS and ONI data

In [3]:
t0 = time.time()
RUTA_IDEAM = DIR_CLIMA / "ideam_resume" / "ideam_historico_diario.csv"
if not RUTA_IDEAM.exists():
    RUTA_IDEAM = RUTA_IDEAM.with_suffix(".csv.gz")
ideam = pd.read_csv(RUTA_IDEAM, dtype={"codigo": str},
                    usecols=["codigo", "estacion", "variable", "fecha", "valor"], parse_dates=["fecha"])
ideam["nombre"] = ideam["estacion"].fillna("cod_" + ideam["codigo"])
INICIO, FIN = pd.Timestamp("2006-01-01"), ideam["fecha"].max()
dias = pd.date_range(INICIO, FIN, freq="D")

niveles = ideam[ideam.variable == "NV_MEDIA_D"].pivot_table(index="fecha", columns="nombre", values="valor").reindex(dias)
caudales = ideam[ideam.variable == "Q_MEDIA_D"].pivot_table(index="fecha", columns="nombre", values="valor").reindex(dias)

glofas = pd.read_csv(DIR_CLIMA / "ideam_resume" / "glofas_caudal_diario.csv", parse_dates=["fecha"])
glofas = glofas.pivot_table(index="fecha", columns="estacion", values="caudal_m3s").reindex(dias)

oni = pd.read_csv(DIR_CLIMA / "ideam_resume" / "oni_mensual.csv", parse_dates=["fecha"]).set_index("fecha")["oni"]
oni = oni.reindex(pd.date_range(oni.index.min(), FIN, freq="D")).ffill().reindex(dias)

cobertura = niveles.notna().mean().sort_values(ascending=False)
EST_NIVEL = [e for e in cobertura.index if cobertura[e] >= 0.60 or e == "Quibdo"]
EST_GLOFAS = ["Quibdo", "Belen_AUT", "El_Siete_AUT", "Bellavista", "Puente_Certegui", "Bagado_AUT"]
print(f"IDEAM {ideam.fecha.min().date()} → {FIN.date()} · {ideam.nombre.nunique()} estaciones")
print("Cobertura 2006-2026 de nivel:", (cobertura.round(2)).to_dict())
print("Estaciones de nivel usadas:", EST_NIVEL)
deriva = pd.DataFrame({"media_2006_2018": niveles[EST_NIVEL].loc["2006":"2018"].mean(),
                       "media_2022_2026": niveles[EST_NIVEL].loc["2022":"2026"].mean()})
deriva["cambio_cm"] = deriva.media_2022_2026 - deriva.media_2006_2018
print("\nCambio del nivel medio entre periodos (cm). Cambios grandes = posible cambio de cero de la mira o del cauce:")
print(deriva.round(0).astype(int).to_string())
print(f"GloFAS {glofas.dropna(how='all').index.min().date()} → {glofas.dropna(how='all').index.max().date()}")
print(f"({time.time() - t0:.1f} s)")

IDEAM 1966-05-01 → 2026-11-13 · 31 estaciones
Cobertura 2006-2026 de nivel: {'Quibdo': 0.99, 'Aguasal': 0.98, 'El_Siete_AUT': 0.98, 'Puente_Certegui': 0.96, 'Bellavista': 0.95, 'cod_0011077020': 0.94, 'Domingodo': 0.92, 'cod_0011117050': 0.91, 'cod_0011117010': 0.9, 'Belen_AUT': 0.9, 'cod_0011057020': 0.86, 'cod_0011047030': 0.85, 'Riosucio': 0.82, 'cod_0011147020': 0.54, 'cod_0011117040': 0.46, 'Boraudo': 0.42, 'cod_0011027050': 0.4, 'cod_0011027010': 0.38, 'La_Loma_Pueblo': 0.3, 'cod_0011047040': 0.29, 'Bagado_AUT': 0.26, 'cod_1111700231': 0.08, 'Curvarado': 0.05, 'cod_0011027040': 0.0, 'cod_0011027020': 0.0, 'cod_0011035020': 0.0, 'cod_0011057010': 0.0, 'cod_0011077030': 0.0, 'cod_0011117020': 0.0, 'cod_0011117030': 0.0, 'cod_0011127010': 0.0}
Estaciones de nivel usadas: ['Quibdo', 'Aguasal', 'El_Siete_AUT', 'Puente_Certegui', 'Bellavista', 'cod_0011077020', 'Domingodo', 'cod_0011117050', 'cod_0011117010', 'Belen_AUT', 'cod_0011057020', 'cod_0011047030', 'Riosucio']

Cambio del nive

## A2. Label: high water at Quibdó in the next *H* days

In [4]:
H = 3
ESTACION = "Quibdo"
q = niveles[ESTACION].ffill(limit=2)
ENTRENA, VALIDA = ("2006-01-01", "2018-12-31"), ("2019-01-01", "2021-12-31")
PRUEBA = ("2022-01-01", str(FIN.date()))
UMBRAL_NIVEL = float(q.loc[ENTRENA[0]:ENTRENA[1]].quantile(0.90))


def etiqueta(serie, h, umbral):
    fut = pd.concat([serie.shift(-k) for k in range(1, h + 1)], axis=1)
    y = (fut.max(axis=1) >= umbral).astype(float)
    y[fut.isna().all(axis=1)] = np.nan
    return y


y_A = etiqueta(q, H, UMBRAL_NIVEL)
print(f"Umbral de nivel alto en {ESTACION}: {UMBRAL_NIVEL:.0f} cm (P90 2006–2018)")
print(f"Días con y=1: {int(y_A.sum())} de {int(y_A.notna().sum())} ({y_A.mean():.1%})")

fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(q.index, q, lw=0.6, color="#3b6ea5")
ax.axhline(UMBRAL_NIVEL, color="#d62728", lw=1, ls="--", label=f"P90 = {UMBRAL_NIVEL:.0f} cm")
ax.axvspan(pd.Timestamp(VALIDA[0]), pd.Timestamp(VALIDA[1]), color="#e3a21a", alpha=0.12, label="validación")
ax.axvspan(pd.Timestamp(PRUEBA[0]), pd.Timestamp(PRUEBA[1]), color="#2ca02c", alpha=0.10, label="prueba")
ax.set_ylabel("nivel (cm)"); ax.set_title("Río Atrato en Quibdó (IDEAM, nivel medio diario)"); ax.legend(loc="upper left", ncol=3, fontsize=8)
plt.tight_layout(); plt.savefig(DIR_OUT / "A_nivel_quibdo.png"); plt.show()

Umbral de nivel alto en Quibdo: 497 cm (P90 2006–2018)
Días con y=1: 1387 de 7557 (18.4%)


## A3. Features

In [5]:
def rasgos_quibdo(s):
    s = s.ffill(limit=2)
    d = {f"q_l{k}": s.shift(k) for k in range(7)}
    d.update(q_d1=s - s.shift(1), q_d3=s - s.shift(3),
             q_m7=s.rolling(7, min_periods=4).mean(), q_x7=s.rolling(7, min_periods=4).max(),
             q_m30=s.rolling(30, min_periods=15).mean())
    return pd.DataFrame(d)


tr = q.loc[ENTRENA[0]:ENTRENA[1]]
clim = tr.groupby(tr.index.dayofyear).mean().reindex(range(1, 367)).interpolate().bfill().ffill()
doy = dias.dayofyear.values
X_q = rasgos_quibdo(q)
X_q["q_anom"] = q - clim.reindex(doy).values
X_q["oni"] = oni.values
X_q["doy_sin"], X_q["doy_cos"] = np.sin(2 * np.pi * doy / 365.25), np.cos(2 * np.pi * doy / 365.25)

X_gf_hoy = pd.concat([pd.DataFrame({f"gf_{e}": glofas[e], f"gf_{e}_d1": glofas[e] - glofas[e].shift(1)})
                      for e in EST_GLOFAS], axis=1)


def gf_pronostico(h):
    return pd.concat([pd.DataFrame({f"gfP_{e}_t{k}": glofas[e].shift(-k) for k in range(1, h + 1)})
                      for e in ("Quibdo", "Belen_AUT", "El_Siete_AUT")], axis=1)


def conjuntos(h):
    return {"1 · Quibdó IDEAM + ONI": X_q,
            "2 · + GloFAS hoy": pd.concat([X_q, X_gf_hoy], axis=1),
            "3 · + pronóstico GloFAS (techo)": pd.concat([X_q, X_gf_hoy, gf_pronostico(h)], axis=1)}


for k, v in conjuntos(H).items():
    print(f"{k:<34} {v.shape[1]:>3} variables")

1 · Quibdó IDEAM + ONI              16 variables
2 · + GloFAS hoy                    28 variables
3 · + pronóstico GloFAS (techo)     37 variables


## A4. Training and comparison of the 6 models

In [6]:
FIN_COMUN = "2026-06-30"


def partir(X, y, fin_prueba=FIN_COMUN, solo_bajo=False):
    ok = y.notna() & q.notna()
    if solo_bajo:
        ok &= q < UMBRAL_NIVEL
    X, y = X[ok], y[ok].astype(int)
    tramo = lambda a, b: (X.index >= pd.Timestamp(a)) & (X.index <= pd.Timestamp(b))
    return [(X[tramo(*r)], y[tramo(*r)]) for r in (ENTRENA, VALIDA, (PRUEBA[0], fin_prueba))]


def comparar(h, tareas=("nivel alto", "inicio de crecida"), nombres_mod=None, guardar=False):
    yh = etiqueta(q, h, UMBRAL_NIVEL)
    filas = []
    for tarea in tareas:
        for nombre_conj, X in conjuntos(h).items():
            (Xe, ye), (Xv, yv), (Xp, yp) = partir(X, yh, solo_bajo=(tarea == "inicio de crecida"))
            base = metricas(yp, Xp["q_l0"].values / UMBRAL_NIVEL / 2, 0.5) if tarea == "nivel alto" else \
                metricas(yp, Xp["q_l0"].values / UMBRAL_NIVEL, umbral_max_f1(yv, Xv["q_l0"].values / UMBRAL_NIVEL))
            base.update(tarea=tarea, conjunto=nombre_conj, modelo="Línea base (nivel de hoy)", H=h, Brier=np.nan,
                        PR_AUC_valida=average_precision_score(yv, Xv["q_l0"]))
            filas.append(base)
            peso = float((ye == 0).sum() / max((ye == 1).sum(), 1))
            for nombre_mod, mod in crear_modelos(peso).items():
                if nombres_mod and nombre_mod not in nombres_mod:
                    continue
                mod.fit(Xe, ye)
                pv, pp = mod.predict_proba(Xv)[:, 1], mod.predict_proba(Xp)[:, 1]
                u = umbral_max_f1(yv, pv)
                m = metricas(yp, pp, u)
                m.update(tarea=tarea, conjunto=nombre_conj, modelo=nombre_mod, H=h,
                         PR_AUC_valida=average_precision_score(yv, pv))
                filas.append(m)
                if guardar:
                    MODELOS_A[(tarea, nombre_conj, nombre_mod)] = {"modelo": mod, "umbral": u, "pv": pv, "yv": yv.values,
                                                                   "pp": pp, "yp": yp.values, "fechas_p": yp.index}
            print(f"  H={h} · {tarea:<17} · {nombre_conj:<32} entrena {len(ye):>5} ({ye.mean():.0%} +) · "
                  f"valida {len(yv)} · prueba {len(yp)} ({yp.mean():.0%} +)")
    return pd.DataFrame(filas)


MODELOS_A = {}
t0 = time.time()
tabla_A = comparar(H, guardar=True)
cols = ["tarea", "conjunto", "modelo", "PR_AUC", "ROC_AUC", "F1", "POD", "FAR", "CSI", "Brier", "PR_AUC_valida"]
tabla_A = tabla_A[cols].sort_values(["tarea", "conjunto", "PR_AUC_valida"], ascending=[False, True, False])
tabla_A.to_csv(DIR_OUT / f"A_comparacion_modelos_H{H}.csv", index=False)
print(f"({time.time() - t0:.0f} s)")
tabla_A.round(3)

  H=3 · nivel alto        · 1 · Quibdó IDEAM + ONI           entrena  4671 (19% +) · valida 1096 · prueba 1642 (17% +)


  H=3 · nivel alto        · 2 · + GloFAS hoy                 entrena  4671 (19% +) · valida 1096 · prueba 1642 (17% +)


  H=3 · nivel alto        · 3 · + pronóstico GloFAS (techo)  entrena  4671 (19% +) · valida 1096 · prueba 1642 (17% +)


  H=3 · inicio de crecida · 1 · Quibdó IDEAM + ONI           entrena  4203 (13% +) · valida 978 · prueba 1489 (13% +)


  H=3 · inicio de crecida · 2 · + GloFAS hoy                 entrena  4203 (13% +) · valida 978 · prueba 1489 (13% +)


  H=3 · inicio de crecida · 3 · + pronóstico GloFAS (techo)  entrena  4203 (13% +) · valida 978 · prueba 1489 (13% +)
(45 s)


,tarea,conjunto,modelo,PR_AUC,ROC_AUC,F1,POD,FAR,CSI,Brier,PR_AUC_valida
0,nivel alto,1 · Quibdó IDEAM + ONI,Línea base (nivel de hoy),0.516,0.772,0.429,0.330,0.386,0.273,NaN,0.512
2,nivel alto,1 · Quibdó IDEAM + ONI,Random Forest,0.490,0.770,0.485,0.558,0.570,0.321,0.129,0.493
1,nivel alto,1 · Quibdó IDEAM + ONI,Regresión logística,0.479,0.772,0.488,0.509,0.531,0.323,0.185,0.475
6,nivel alto,1 · Quibdó IDEAM + ONI,Red neuronal (MLP),0.471,0.776,0.471,0.653,0.631,0.308,0.119,0.457
5,nivel alto,1 · Quibdó IDEAM + ONI,LightGBM,0.429,0.736,0.430,0.572,0.656,0.273,0.142,0.441
4,nivel alto,1 · Quibdó IDEAM + ONI,XGBoost,0.437,0.739,0.433,0.540,0.639,0.276,0.149,0.436
3,nivel alto,1 · Quibdó IDEAM + ONI,HistGradientBoosting,0.410,0.718,0.409,0.446,0.622,0.257,0.145,0.408
7,nivel alto,2 · + GloFAS hoy,Línea base (nivel de hoy),0.516,0.772,0.429,0.330,0.386,0.273,NaN,0.512
9,nivel alto,2 · + GloFAS hoy,Random Forest,0.478,0.776,0.492,0.519,0.533,0.326,0.128,0.486
8,nivel alto,2 · + GloFAS hoy,Regresión logística,0.468,0.771,0.477,0.646,0.622,0.313,0.189,0.474


### Comparison plot

In [7]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4), sharex=False)
for ax, tarea in zip(axs, ["nivel alto", "inicio de crecida"]):
    t = tabla_A[tabla_A.tarea == tarea].pivot_table(index="modelo", columns="conjunto", values="PR_AUC")
    t = t.loc[t.mean(axis=1).sort_values().index]
    t.plot.barh(ax=ax, width=0.8, color=["#8fb3d9", "#3b6ea5", "#c9a227"])
    ax.set_title(f"{tarea} · H={H} días"); ax.set_xlabel("PR-AUC en prueba (2022–2026)"); ax.set_ylabel("")
    ax.legend(fontsize=7, loc="lower right")
plt.tight_layout(); plt.savefig(DIR_OUT / "A_comparacion_pr_auc.png"); plt.show()

## A5. Lead time: H = 1, 3, 5 and 7 days

In [8]:
t0 = time.time()
tabla_H = pd.concat([comparar(h, tareas=("inicio de crecida",),
                              nombres_mod=("Regresión logística", "Random Forest", "LightGBM", "Red neuronal (MLP)"))
                     for h in (1, 3, 5, 7)])
tabla_H = tabla_H[tabla_H.conjunto != "3 · + pronóstico GloFAS (techo)"]
tabla_H.to_csv(DIR_OUT / "A_anticipacion.csv", index=False)
fig, ax = plt.subplots(figsize=(7, 3.5))
for (conj, mod), g in tabla_H[tabla_H.conjunto == "2 · + GloFAS hoy"].groupby(["conjunto", "modelo"]):
    ax.plot(g.H, g.PR_AUC, marker="o", label=mod)
ax.set_xlabel("días de anticipación (H)"); ax.set_ylabel("PR-AUC en prueba"); ax.set_xticks([1, 3, 5, 7])
ax.set_title("Inicio de crecida en Quibdó · conjunto 2"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(DIR_OUT / "A_anticipacion.png"); plt.show()
print(f"({time.time() - t0:.0f} s)")
tabla_H.pivot_table(index=["conjunto", "modelo"], columns="H", values="PR_AUC").round(3)

  H=1 · inicio de crecida · 1 · Quibdó IDEAM + ONI           entrena  4202 (5% +) · valida 978 · prueba 1489 (5% +)


  H=1 · inicio de crecida · 2 · + GloFAS hoy                 entrena  4202 (5% +) · valida 978 · prueba 1489 (5% +)


  H=1 · inicio de crecida · 3 · + pronóstico GloFAS (techo)  entrena  4202 (5% +) · valida 978 · prueba 1489 (5% +)


  H=3 · inicio de crecida · 1 · Quibdó IDEAM + ONI           entrena  4203 (13% +) · valida 978 · prueba 1489 (13% +)


  H=3 · inicio de crecida · 2 · + GloFAS hoy                 entrena  4203 (13% +) · valida 978 · prueba 1489 (13% +)


  H=3 · inicio de crecida · 3 · + pronóstico GloFAS (techo)  entrena  4203 (13% +) · valida 978 · prueba 1489 (13% +)


... (3 outputs left out)

  H=7 · inicio de crecida · 1 · Quibdó IDEAM + ONI           entrena  4204 (27% +) · valida 978 · prueba 1489 (26% +)


  H=7 · inicio de crecida · 2 · + GloFAS hoy                 entrena  4204 (27% +) · valida 978 · prueba 1489 (26% +)


  H=7 · inicio de crecida · 3 · + pronóstico GloFAS (techo)  entrena  4204 (27% +) · valida 978 · prueba 1489 (26% +)
(37 s)


H                                                     1      3      5      7
conjunto               modelo                                               
1 · Quibdó IDEAM + ONI LightGBM                   0.127  0.215  0.264  0.308
                       Línea base (nivel de hoy)  0.211  0.292  0.331  0.391
                       Random Forest              0.155  0.232  0.290  0.348
                       Red neuronal (MLP)         0.038  0.222  0.307  0.382
                       Regresión logística        0.201  0.246  0.297  0.361
2 · + GloFAS hoy       LightGBM                   0.133  0.213  0.272  0.324
                       Línea base (nivel de hoy)  0.211  0.292  0.331  0.391
                       Random Forest              0.156  0.246  0.308  0.375
                       Red neuronal (MLP)         0.037  0.225  0.303  0.400
                       Regresión logística        0.215  0.270  0.320  0.373

## A6. Selected model, calibration and check against the 2026 events

In [9]:
def elegir(tarea, conjunto):
    t = tabla_A[(tabla_A.tarea == tarea) & (tabla_A.conjunto == conjunto) & (~tabla_A.modelo.str.startswith("Línea"))]
    return t.sort_values("PR_AUC_valida", ascending=False).iloc[0]["modelo"]


ELEGIDOS = {}
for clave, conj in [("operativo", "2 · + GloFAS hoy"), ("con_pronostico", "3 · + pronóstico GloFAS (techo)"),
                    ("solo_ideam", "1 · Quibdó IDEAM + ONI")]:
    nombre = elegir("nivel alto", conj)
    r = MODELOS_A[("nivel alto", conj, nombre)]
    iso = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(r["pv"], r["yv"])
    pp_cal = iso.predict(r["pp"])
    u_cal = umbral_max_f1(r["yv"], iso.predict(r["pv"]))
    m = metricas(r["yp"], pp_cal, u_cal)
    variables = list(conjuntos(H)[conj].columns)
    paquete = {"modelo": r["modelo"], "calibrador": iso, "umbral_decision": u_cal, "variables": variables,
               "H_dias": H, "estacion": ESTACION, "umbral_nivel_cm": UMBRAL_NIVEL, "conjunto": conj, "nombre": nombre,
               "entrenado": ENTRENA, "validado": VALIDA, "metricas_prueba": m,
               "climatologia_doy": clim.values.tolist(), "generado_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}
    joblib.dump(paquete, DIR_OUT / f"modeloA_{clave}.joblib")
    ELEGIDOS[clave] = paquete
    print(f"{clave:<15} {nombre:<22} {conj:<32} PR-AUC {m['PR_AUC']:.3f} · ROC {m['ROC_AUC']:.3f} · CSI {m['CSI']:.3f} · "
          f"Brier calibrado {m['Brier']:.3f} · umbral {u_cal:.2f}")

pk = ELEGIDOS["solo_ideam"]
Xs = conjuntos(H)[pk["conjunto"]][pk["variables"]]
ok = q.notna()
prob = pd.Series(pk["calibrador"].predict(pk["modelo"].predict_proba(Xs[ok])[:, 1]), index=Xs[ok].index)
clima_ev = pd.read_csv(DIR_CLIMA / "condiciones" / "2_datos_nasa_clima_eventos.csv", parse_dates=["fecha"])
eventos = clima_ev.groupby("evento_id").agg(lugar=("municipio_id", "first"), fecha=("fecha", "min"),
                                           lluvia_7d_max_mm=("lluvia_7d_mm", "max"), humedad_raiz=("humedad_raiz_nasa", "max"))
filas = []
for ev, r in eventos.iterrows():
    f = r.fecha
    filas.append({"evento": ev, "lugar": r.lugar, "fecha": f.date(), "lluvia_7d_max_mm": r.lluvia_7d_max_mm,
                  "prob_max_3d_antes": prob.loc[f - pd.Timedelta(days=3): f - pd.Timedelta(days=1)].max(),
                  "nivel_max_Quibdo_cm_en_evento": q.loc[f: f + pd.Timedelta(days=3)].max(),
                  "supera_P90": bool(q.loc[f: f + pd.Timedelta(days=3)].max() >= UMBRAL_NIVEL)})
verif = pd.DataFrame(filas)
verif.to_csv(DIR_OUT / "A_verificacion_eventos_2026.csv", index=False)

fig, ax1 = plt.subplots(figsize=(11, 3.4))
ax1.plot(q.loc["2026-01-01":].index, q.loc["2026-01-01":], color="#3b6ea5", lw=1, label="nivel Quibdó")
ax1.axhline(UMBRAL_NIVEL, color="#d62728", ls="--", lw=0.8)
ax2 = ax1.twinx(); ax2.fill_between(prob.loc["2026-01-01":].index, prob.loc["2026-01-01":], color="#c9a227", alpha=0.35, label=f"prob. nivel alto en {H} d")
ax2.set_ylim(0, 1); ax2.set_ylabel("probabilidad")
for _, r in verif.iterrows():
    ax1.axvline(pd.Timestamp(r.fecha), color="k", lw=0.6, ls=":"); ax1.text(pd.Timestamp(r.fecha), ax1.get_ylim()[1] * 0.97, r.evento, fontsize=8)
ax1.set_ylabel("nivel (cm)"); ax1.set_title("2026: nivel observado, probabilidad del modelo y eventos E1–E6")
plt.tight_layout(); plt.savefig(DIR_OUT / "A_eventos_2026.png"); plt.show()
verif.round(2)

operativo       Random Forest          2 · + GloFAS hoy                 PR-AUC 0.443 · ROC 0.767 · CSI 0.326 · Brier calibrado 0.118 · umbral 0.33
con_pronostico  XGBoost                3 · + pronóstico GloFAS (techo)  PR-AUC 0.458 · ROC 0.790 · CSI 0.314 · Brier calibrado 0.118 · umbral 0.21
solo_ideam      Random Forest          1 · Quibdó IDEAM + ONI           PR-AUC 0.450 · ROC 0.766 · CSI 0.321 · Brier calibrado 0.118 · umbral 0.33


,evento,lugar,fecha,lluvia_7d_max_mm,prob_max_3d_antes,nivel_max_Quibdo_cm_en_evento,supera_P90
0,E1,Murindo,2026-01-19,110.0,0.43,510.02,True
1,E2,Vigia_del_Fuerte,2026-02-01,167.0,0.22,468.67,False
2,E3,Rio_Quito,2026-05-04,232.6,0.09,511.69,True
3,E4,Quibdo,2026-05-29,160.8,0.33,544.75,True
4,E5,Medio_Atrato,2026-07-19,122.1,0.09,252.50,False
5,E6,Quibdo,2026-08-28,310.0,0.09,477.50,False


# Part B · NISAR spatial model: where does it flood?
## B0. (Optional) Downsample the subsets to 40 m

In [10]:
REDUCIR = False
DIR_RECORTES = DIR_ML.parent / "recortes"


def reducir_recortes(dir_recortes, dir_salida):
    dir_salida.mkdir(parents=True, exist_ok=True)
    for r in sorted(Path(dir_recortes).glob("[AD]_T*/2026-*/*.npz")):
        stack, fecha = r.name[:6], r.name[7:15]
        dst = dir_salida / f"{stack}_{fecha}_sar40.npz"
        if dst.exists():
            continue
        res = {}
        with np.load(r) as z:
            for b in ("HH", "HV"):
                if b in z.files:
                    a = z[b].astype(np.float32)
                    ny, nx = a.shape[0] // 2 * 2, a.shape[1] // 2 * 2
                    with np.errstate(all="ignore"):
                        m = np.nanmean(a[:ny, :nx].reshape(ny // 2, 2, nx // 2, 2), axis=(1, 3))
                        db = 10 * np.log10(np.maximum(m, 1e-3))
                    db[~np.isfinite(m)] = np.nan
                    res[b] = db.astype(np.float16)
        np.savez_compressed(dst, **res)
        print("ok", dst.name)


if REDUCIR:
    reducir_recortes(DIR_RECORTES, DIR_INS / "sar40")

## B1. Terrain, classes and radar on the 40 m grid

In [11]:
import rasterio
from rasterio.transform import from_origin
from scipy import ndimage
from sklearn.model_selection import GroupKFold

X0, Y1, NY20, NX20 = 272_880, 796_440, 9144, 3351
NY, NX, RES = NY20 // 2, NX20 // 2, 40
TRANSF40 = from_origin(X0, Y1, RES, RES)


def a40_media(a):
    a = a[:NY * 2, :NX * 2].astype(np.float32).reshape(NY, 2, NX, 2)
    with np.errstate(all="ignore"):
        return np.nanmean(a, axis=(1, 3))


def leer(ruta):
    with rasterio.open(ruta) as r:
        return r.read(1), r.transform, r.nodata


t0 = time.time()
TER = {}
for nombre in ("HAND", "DEM", "PEND"):
    a, _, nd = leer(DIR_INS / "terreno" / f"{nombre}.tif")
    TER[nombre] = a40_media(np.where(a == nd, np.nan, a) if nd is not None and not np.isnan(nd) else a)
wc, _, _ = leer(DIR_INS / "terreno" / "WORLDCOVER.tif")
TER["WC"] = wc[:NY * 2:2, :NX * 2:2]
print({k: v.shape for k, v in TER.items()}, f"({time.time() - t0:.0f} s)")

FECHAS = {}
for f in sorted((DIR_INS / "clases").glob("*_clases_solo_nisar.tif")):
    stack, fecha = f.name[:6], pd.Timestamp(f.name[7:15])
    if (DIR_INS / "sar40" / f"{stack}_{fecha:%Y%m%d}_sar40.npz").exists():
        FECHAS.setdefault(stack, []).append(fecha)
for s in FECHAS:
    FECHAS[s] = sorted(FECHAS[s])
    print(s, [d.strftime("%m-%d") for d in FECHAS[s]])


def clases40(stack, fecha):
    a, t, _ = leer(DIR_INS / "clases" / f"{stack}_{fecha:%Y%m%d}_clases_solo_nisar.tif")
    full = np.full((NY20, NX20), 255, np.uint8)
    r0, c0 = int(round((Y1 - t.f) / 20)), int(round((t.c - X0) / 20))
    full[r0:r0 + a.shape[0], c0:c0 + a.shape[1]] = a
    b = full[:NY * 2, :NX * 2].reshape(NY, 2, NX, 2)
    cnt = lambda ks: np.isin(b, ks).sum(axis=(1, 3)).astype(np.uint8)
    return {"valido": cnt(range(1, 10)), "c1": cnt([1]), "c2": cnt([2]), "c3": cnt([3]), "c4": cnt([4]),
            "c78": cnt([7, 8]), "c9": cnt([9])}


def sar40(stack, fecha):
    with np.load(DIR_INS / "sar40" / f"{stack}_{fecha:%Y%m%d}_sar40.npz") as z:
        hh = z["HH"].astype(np.float32)
        hv = z["HV"].astype(np.float32) if "HV" in z.files else np.full_like(hh, np.nan)
    return hh, hv

{'HAND': (4572, 1675), 'DEM': (4572, 1675), 'PEND': (4572, 1675), 'WC': (4572, 1675)} (4 s)
A_T047 ['06-17', '07-11', '07-23', '08-16', '08-28', '09-09', '09-21']
A_T148 ['06-24', '07-06', '08-11', '09-04', '09-16']
D_T011 ['06-26', '07-08', '07-20', '08-13', '09-06', '09-18']
D_T112 ['06-21', '08-20', '09-01', '09-25']


## B2. Hydrological scenario for each pair (IDEAM Quibdó)

In [12]:
def escenario(t, t1):
    tramo = q.loc[t:t1]
    return {"nivel_t": float(q.get(t, np.nan)), "nivel_t1": float(q.get(t1, np.nan)),
            "nivel_max_tramo": float(tramo.max()), "dnivel": float(q.get(t1, np.nan) - q.get(t, np.nan)),
            "nivel_max_rel_P90": float(tramo.max() / UMBRAL_NIVEL), "dias": int((t1 - t).days),
            "oni_t1": float(oni.get(t1, np.nan))}


PARES = [(s, FECHAS[s][i], FECHAS[s][i + 1]) for s in FECHAS for i in range(len(FECHAS[s]) - 1)]
tabla_pares = pd.DataFrame([{"stack": s, "t": t.date(), "t1": t1.date(), **escenario(t, t1)} for s, t, t1 in PARES])
tabla_pares.round(2)

,stack,t,t1,nivel_t,nivel_t1,nivel_max_tramo,dnivel,nivel_max_rel_P90,dias,oni_t1
0,A_T047,2026-06-17,2026-07-11,260.88,197.19,432.81,-63.69,0.87,24,1.80
1,A_T047,2026-07-11,2026-07-23,197.19,201.87,296.25,4.69,0.60,12,1.80
2,A_T047,2026-07-23,2026-08-16,201.87,275.62,329.06,73.75,0.66,24,2.16
3,A_T047,2026-08-16,2026-08-28,275.62,333.94,446.25,58.31,0.90,12,2.16
4,A_T047,2026-08-28,2026-09-09,333.94,139.06,477.50,-194.87,0.96,12,2.46
5,A_T047,2026-09-09,2026-09-21,139.06,338.50,435.94,199.44,0.88,12,2.46
6,A_T148,2026-06-24,2026-07-06,360.19,117.12,360.19,-243.06,0.72,12,1.80
7,A_T148,2026-07-06,2026-08-11,117.12,141.25,329.06,24.13,0.66,36,2.16
8,A_T148,2026-08-11,2026-09-04,141.25,425.31,477.50,284.06,0.96,24,2.46
9,A_T148,2026-09-04,2026-09-16,425.31,285.63,435.94,-139.69,0.88,12,2.46


## B3. Building the training set

In [13]:
VAR_B = None
rng = np.random.default_rng(SEMILLA)


def rasgos_par(stack, t, t1, cache, todo=False):
    hh, hv = cache["sar"][t]
    pasadas = [d for d in FECHAS[stack] if d <= t]
    with np.errstate(all="ignore"):
        pila = np.stack([cache["sar"][d][0] for d in pasadas])
        hh_ref, hh_min, hh_max = np.nanmean(pila, 0), np.nanmin(pila, 0), np.nanmax(pila, 0)
    c, c1 = cache["cl"][t], cache["cl"][t1]
    vt = np.maximum(c["valido"], 1).astype(np.float32)
    agua = (c["c1"] + c["c3"]) >= 2
    nueva = (c["c2"] + c["c4"]) >= 2
    d_agua = ndimage.distance_transform_edt(~agua) * RES / 1000 if agua.any() else np.full(agua.shape, 99.0)
    d_nueva = ndimage.distance_transform_edt(~nueva) * RES / 1000 if nueva.any() else np.full(agua.shape, 99.0)
    ok = (c["valido"] >= 2) & (c1["valido"] >= 2) & np.isfinite(hh)
    y = ((c1["c2"] + c1["c4"]) >= 2) & ok
    esc = escenario(t, t1)
    if todo:
        idx = np.flatnonzero(ok.ravel())
    else:
        pos = np.flatnonzero(y.ravel())
        pos = rng.choice(pos, min(len(pos), 15000), replace=False) if len(pos) else pos
        n_neg = max(6 * len(pos), 15000)
        neg_cerca = np.flatnonzero((ok & ~y & (d_agua <= 3)).ravel())
        neg_lejos = np.flatnonzero((ok & ~y & (d_agua > 3)).ravel())
        idx = np.concatenate([pos, rng.choice(neg_cerca, min(len(neg_cerca), n_neg // 2), replace=False),
                              rng.choice(neg_lejos, min(len(neg_lejos), n_neg // 2), replace=False)])
    filas, cols = np.unravel_index(idx, (NY, NX))
    g = lambda a: a.ravel()[idx]
    wcv = g(TER["WC"])
    df = pd.DataFrame({
        "HH": g(hh), "HV": g(hv), "HH_HV": g(hh - hv), "HH_menos_ref": g(hh - hh_ref), "HH_min": g(hh_min), "HH_max": g(hh_max),
        "f_agua_perm": g(c["c1"] / vt), "f_agua_nueva": g(c["c2"] / vt), "f_veg_inund_perm": g(c["c3"] / vt),
        "f_veg_inund_nueva": g(c["c4"] / vt), "f_urbano": g(c["c78"] / vt), "f_dudoso": g(c["c9"] / vt),
        "d_agua_km": g(d_agua), "d_inund_nueva_km": g(d_nueva),
        "HAND": g(TER["HAND"]), "DEM": g(TER["DEM"]), "PEND": g(TER["PEND"]),
        "wc_bosque": (wcv == 10).astype(np.int8), "wc_herb": np.isin(wcv, [20, 30, 40]).astype(np.int8),
        "wc_urbano": (wcv == 50).astype(np.int8), "wc_agua": (wcv == 80).astype(np.int8), "wc_humedal": np.isin(wcv, [90, 95]).astype(np.int8),
        "ascendente": int(stack.startswith("A")),
        **{k: v for k, v in esc.items()},
    })
    df["y"] = g(y).astype(np.int8)
    df["stack"], df["t"], df["t1"] = stack, t, t1
    df["fila"], df["col"] = filas, cols
    df["bloque"] = (filas // 250) * 100 + cols // 250
    return df


t0 = time.time()
partes = []
for stack in FECHAS:
    cache = {"sar": {d: sar40(stack, d) for d in FECHAS[stack]}, "cl": {d: clases40(stack, d) for d in FECHAS[stack]}}
    for s, t, t1 in [p for p in PARES if p[0] == stack]:
        d = rasgos_par(s, t, t1, cache)
        partes.append(d)
        print(f"{s} {t:%m-%d} → {t1:%m-%d}: {int(d.y.sum()):>6} positivas · {len(d):>6} filas")
    del cache
datos_B = pd.concat(partes, ignore_index=True)
VAR_B = [c for c in datos_B.columns if c not in ("y", "stack", "t", "t1", "fila", "col", "bloque")]
print(f"Total: {len(datos_B):,} filas · {int(datos_B.y.sum()):,} positivas ({datos_B.y.mean():.1%}) · {len(VAR_B)} variables · "
      f"({time.time() - t0:.0f} s)")

A_T047 06-17 → 07-11:      0 positivas ·  15000 filas


A_T047 07-11 → 07-23:      0 positivas ·   7500 filas


A_T047 07-23 → 08-16:      0 positivas ·   7500 filas


A_T047 08-16 → 08-28:      0 positivas ·   7500 filas


A_T047 08-28 → 09-09:      0 positivas ·   7500 filas


A_T047 09-09 → 09-21:      0 positivas ·   7500 filas


... (8 outputs left out)

D_T011 09-06 → 09-18:    724 positivas ·  15724 filas


D_T112 06-21 → 08-20:   3888 positivas ·  27216 filas


D_T112 08-20 → 09-01:   2300 positivas ·  17300 filas


D_T112 09-01 → 09-25:    165 positivas ·  15165 filas
Total: 329,679 filas · 25,287 positivas (7.7%) · 30 variables · (59 s)


## B4. Model comparison

In [14]:
def lineas_base(df):
    return {"Persistencia (inund. nueva en t)": (df.f_agua_nueva + df.f_veg_inund_nueva).values,
            "Regla de terreno (HAND y d_agua)": (1 / (1 + df.HAND.fillna(50).clip(0))) * np.exp(-df.d_agua_km.values / 2)}


def modelos_B(peso):
    m = crear_modelos(peso)
    m["Random Forest"].set_params(randomforestclassifier__n_estimators=150, randomforestclassifier__max_samples=0.3,
                                  randomforestclassifier__min_samples_leaf=10)
    m["Red neuronal (MLP)"].set_params(mlpclassifier__max_iter=60)
    m["XGBoost"].set_params(n_estimators=400, max_depth=7, tree_method="hist")
    m["LightGBM"].set_params(n_estimators=400, num_leaves=63)
    m["HistGradientBoosting"].set_params(max_iter=300)
    return m


UMBRAL_B = 0.5
t0 = time.time()
filas = []
# ---- 1) Spatial validation ----
gkf = GroupKFold(n_splits=3)
oof = {n: np.zeros(len(datos_B)) for n in list(modelos_B(1)) + list(lineas_base(datos_B.head()))}
for k, (ie, ip) in enumerate(gkf.split(datos_B, datos_B.y, datos_B.bloque)):
    de, dp = datos_B.iloc[ie], datos_B.iloc[ip]
    peso = float((de.y == 0).sum() / max((de.y == 1).sum(), 1))
    for nombre, mod in modelos_B(peso).items():
        mod.fit(de[VAR_B], de.y)
        oof[nombre][ip] = mod.predict_proba(dp[VAR_B])[:, 1]
    for nombre, s in lineas_base(dp).items():
        oof[nombre][ip] = s
    print(f"  pliegue espacial {k + 1}/3 listo ({time.time() - t0:.0f} s)")
for nombre, p in oof.items():
    es_base = nombre not in modelos_B(1)
    m = metricas(datos_B.y.values, p, umbral_max_f1(datos_B.y.values, p) if es_base else UMBRAL_B)
    m.update(validacion="espacial (bloques 10 km)", modelo=nombre); filas.append(m)

# ---- 2) Event validation (E6) ----
CORTE = pd.Timestamp("2026-08-20")
de_all, dp = datos_B[datos_B.t1 < CORTE], datos_B[datos_B.t1 >= CORTE]
bl = de_all.bloque.unique(); bl_val = set(rng.choice(bl, max(1, len(bl) // 5), replace=False))
dv, de = de_all[de_all.bloque.isin(bl_val)], de_all[~de_all.bloque.isin(bl_val)]
peso = float((de.y == 0).sum() / max((de.y == 1).sum(), 1))
MODELOS_B = {}
for nombre, mod in modelos_B(peso).items():
    mod.fit(de[VAR_B], de.y)
    u = UMBRAL_B
    m = metricas(dp.y.values, mod.predict_proba(dp[VAR_B])[:, 1], u)
    m.update(validacion="por evento (prueba E6)", modelo=nombre); filas.append(m)
    MODELOS_B[nombre] = (mod, u)
for nombre, s in lineas_base(dp).items():
    sv = lineas_base(dv)[nombre]
    m = metricas(dp.y.values, s, umbral_max_f1(dv.y.values, sv))
    m.update(validacion="por evento (prueba E6)", modelo=nombre); filas.append(m)
print(f"Evento: entrena {len(de):,} · umbral {len(dv):,} · prueba E6 {len(dp):,} ({dp.y.mean():.1%} +) · ({time.time() - t0:.0f} s)")

tabla_B = pd.DataFrame(filas)[["validacion", "modelo", "PR_AUC", "ROC_AUC", "F1", "POD", "FAR", "CSI", "Brier", "umbral"]]
tabla_B = tabla_B.sort_values(["validacion", "PR_AUC"], ascending=[False, False])
tabla_B.to_csv(DIR_OUT / "B_comparacion_modelos.csv", index=False)
tabla_B.round(3)

  pliegue espacial 1/3 listo (70 s)


  pliegue espacial 2/3 listo (138 s)


  pliegue espacial 3/3 listo (202 s)


Evento: entrena 86,548 · umbral 19,704 · prueba E6 223,427 (10.8% +) · (233 s)


,validacion,modelo,PR_AUC,ROC_AUC,F1,POD,FAR,CSI,Brier,umbral
11,por evento (prueba E6),XGBoost,0.622,0.934,0.263,0.155,0.149,0.151,0.087,0.500
10,por evento (prueba E6),HistGradientBoosting,0.614,0.928,0.315,0.194,0.159,0.187,0.086,0.500
9,por evento (prueba E6),Random Forest,0.583,0.932,0.060,0.031,0.296,0.031,0.072,0.500
12,por evento (prueba E6),LightGBM,0.577,0.902,0.106,0.056,0.041,0.056,0.099,0.500
8,por evento (prueba E6),Regresión logística,0.536,0.872,0.407,0.278,0.236,0.256,0.080,0.500
15,por evento (prueba E6),Regla de terreno (HAND y d_agua),0.406,0.891,0.295,0.206,0.479,0.173,0.088,0.945
13,por evento (prueba E6),Red neuronal (MLP),0.328,0.777,0.162,0.090,0.225,0.088,0.097,0.500
14,por evento (prueba E6),Persistencia (inund. nueva en t),0.145,0.523,0.194,1.000,0.892,0.108,0.104,0.000
4,espacial (bloques 10 km),LightGBM,0.880,0.989,0.793,0.897,0.289,0.657,0.027,0.500
2,espacial (bloques 10 km),HistGradientBoosting,0.877,0.988,0.775,0.926,0.333,0.633,0.030,0.500


### Comparison plot and feature importance

In [15]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
t = tabla_B.pivot_table(index="modelo", columns="validacion", values="PR_AUC")
t = t.loc[t.mean(axis=1).sort_values().index]
t.plot.barh(ax=axs[0], width=0.8, color=["#3b6ea5", "#c9a227"])
axs[0].set_xlabel("PR-AUC"); axs[0].set_ylabel(""); axs[0].set_title("Modelo B · inundación nueva en la pasada siguiente"); axs[0].legend(fontsize=8)
MEJOR_B = tabla_B[(tabla_B.validacion.str.startswith("por evento")) & tabla_B.modelo.isin(MODELOS_B)].iloc[0]["modelo"]
mod_b = MODELOS_B[MEJOR_B][0]
imp = getattr(mod_b, "feature_importances_", None)
if imp is None:
    from sklearn.inspection import permutation_importance
    muestra = dp.sample(min(20000, len(dp)), random_state=SEMILLA)
    imp = permutation_importance(mod_b, muestra[VAR_B], muestra.y, n_repeats=3, random_state=SEMILLA, scoring="average_precision").importances_mean
imp = pd.Series(imp, index=VAR_B).sort_values().tail(15)
(imp / imp.sum()).plot.barh(ax=axs[1], color="#3b6ea5"); axs[1].set_title(f"Importancia de variables · {MEJOR_B}")
plt.tight_layout(); plt.savefig(DIR_OUT / "B_comparacion_e_importancia.png"); plt.show()
print("Mejor modelo B (prueba por evento):", MEJOR_B)

Mejor modelo B (prueba por evento): XGBoost


## B5. Probability map for V.I.S.O.R. (E6 · A_T148, Aug 11 → Sep 4)

In [16]:
STACK_DEMO, T_DEMO, T1_DEMO = "A_T148", pd.Timestamp("2026-08-11"), pd.Timestamp("2026-09-04")
cache = {"sar": {d: sar40(STACK_DEMO, d) for d in FECHAS[STACK_DEMO]},
         "cl": {d: clases40(STACK_DEMO, d) for d in FECHAS[STACK_DEMO]}}
todo = rasgos_par(STACK_DEMO, T_DEMO, T1_DEMO, cache, todo=True)
p_todo = np.zeros(len(todo), np.float32)
for i in range(0, len(todo), 500_000):
    p_todo[i:i + 500_000] = mod_b.predict_proba(todo[VAR_B].iloc[i:i + 500_000])[:, 1]
mapa = np.full((NY, NX), 255, np.uint8)
mapa[todo.fila.values, todo.col.values] = np.clip(np.round(p_todo * 100), 0, 100).astype(np.uint8)
real = todo.y.values == 1

def captura(puntaje, frac):
    k = int(len(puntaje) * frac)
    return float(real[np.argpartition(-puntaje, k)[:k]].sum() / max(real.sum(), 1))


eval_mapa = pd.DataFrame([{"puntaje": n, "PR_AUC": average_precision_score(real, sc), "ROC_AUC": roc_auc_score(real, sc),
                           **{f"captura_top_{int(f * 100)}%": captura(np.asarray(sc, float), f) for f in (0.01, 0.05, 0.10)}}
                          for n, sc in [(MEJOR_B, p_todo)] + list(lineas_base(todo).items())])
eval_mapa.to_csv(DIR_OUT / "B_mapa_E6_evaluacion.csv", index=False)
u_b = float(np.quantile(p_todo, 0.95))
pred = p_todo >= u_b
iou = (pred & real).sum() / max((pred | real).sum(), 1)
m_demo = metricas(real.astype(int), p_todo, u_b)
print(f"Celdas evaluadas: {len(todo):,} · inundación nueva (etiquetado): {real.sum() * 0.0016:.1f} km² ({real.mean():.2%}) · "
      f"zona de alerta (5 % más probable): {pred.sum() * 0.0016:.0f} km² · umbral {u_b:.2f}")
print(eval_mapa.round(3).to_string())

nombre_tif = f"CHOCO_{STACK_DEMO}_E6_prob_inundacion_{T1_DEMO:%Y%m%d}.tif"
perfil = {"driver": "GTiff", "height": NY, "width": NX, "count": 1, "dtype": "uint8", "crs": "EPSG:32618",
          "transform": TRANSF40, "nodata": 255, "compress": "deflate", "tiled": True}
with rasterio.open(DIR_OUT / nombre_tif, "w", **perfil) as dst:
    dst.write(mapa, 1)
    dst.update_tags(TIPO_PRODUCTO="probabilidad_inundacion_nisar", UNIDADES="porcentaje", FECHA=f"{T1_DEMO:%Y-%m-%d}",
                    SENSOR="NISAR L-SAR banda L", VERSION_MODELO=f"modeloB_{MEJOR_B}")
sha = hashlib.sha256((DIR_OUT / nombre_tif).read_bytes()).hexdigest()
filas_v, cols_v = np.nonzero(mapa != 255)
ficha = {
    "esquema": "visor.ficha.v1",
    "tipo_producto": "probabilidad_inundacion_nisar",
    "producto": "Probabilidad de inundación nueva en la pasada siguiente (modelo B)",
    "fuente": "NISAR L2 GCOV (ASF DAAC) + IDEAM + Copernicus DEM/HAND + ESA WorldCover",
    "fecha": f"{T1_DEMO:%Y-%m-%d}", "unidades": "porcentaje (0-100)", "nodata": 255,
    "advertencia": "Pronóstico experimental entrenado con etiquetas no supervisadas (sin verdad de campo). "
                   "Indica dónde es más probable que aparezca agua nueva si se da el escenario hidrológico indicado.",
    "procedencia": {
        "estado": "real_sin_validar",
        "sensor": f"NISAR L-SAR (banda L), stack {STACK_DEMO}",
        "fechas": [f"{T_DEMO:%Y-%m-%d}", f"{T1_DEMO:%Y-%m-%d}"],
        "polarizaciones": ["HH", "HV"],
        "metodo": f"{MEJOR_B} sobre {len(VAR_B)} variables (radar en t, clases en t, terreno, escenario IDEAM t→t+1); "
                  f"entrenado sin el evento E6; umbral de alerta {u_b:.2f}",
        "entradas": [{"papel": "estado", "fecha": f"{T_DEMO:%Y-%m-%d}"}, {"papel": "escenario", "nivel_max_cm": escenario(T_DEMO, T1_DEMO)["nivel_max_tramo"]}],
        "generado_por": {"modulo": "ml_choco", "notebook": "entrenamiento_prediccion_inundaciones.ipynb"},
        "generado_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "archivo": {"nombre": nombre_tif, "sha256": sha},
        "validacion": None,
    },
    "fechas_detalle": {"observacion": f"{T_DEMO:%Y-%m-%d}", "fecha_objetivo": f"{T1_DEMO:%Y-%m-%d}",
                       "nota": "Fecha objetivo = pasada siguiente. La prueba se hizo con E6 fuera del entrenamiento."},
    "umbral_alerta_pct": round(u_b * 100),
    "criterio_umbral": "percentil 95 de la franja: la alerta marca el 5 % de celdas más probables",
    "metricas_prueba_E6": {k: round(float(v), 3) for k, v in m_demo.items()} | {"IoU": round(float(iou), 3)}
                          | {k: round(float(v), 3) for k, v in eval_mapa.iloc[0].items() if k.startswith("captura")},
    "cuadricula": {"crs": "EPSG:32618", "resolucion": [RES, RES],
                   "limites": [X0, Y1 - NY * RES, X0 + NX * RES, Y1], "ancho": NX, "alto": NY},
    "limitaciones": ["Etiquetas del etiquetado no supervisado solo NISAR, no de campo.",
                     "Productos NISAR provisionales (bandeo de 1-2 dB).",
                     "Bajo bosque muy denso la inundación se subestima.",
                     "Solo 18 pares de fechas (jun-sep 2026) y un escenario hidrológico por par."],
}
(DIR_OUT / nombre_tif.replace(".tif", ".json")).write_text(json.dumps(ficha, ensure_ascii=False, indent=2), encoding="utf-8")

r0, r1, c0, c1 = filas_v.min(), filas_v.max(), cols_v.min(), cols_v.max()
real_map = np.full((NY, NX), np.nan); real_map[todo.fila.values, todo.col.values] = real
prob_map = np.where(mapa == 255, np.nan, mapa / 100.0)
fig, axs = plt.subplots(1, 3, figsize=(13, 6), sharey=True)
hh_t1 = cache["sar"][T1_DEMO][0]
axs[0].imshow(hh_t1[r0:r1, c0:c1], cmap="gray", vmin=-25, vmax=0); axs[0].set_title(f"HH {T1_DEMO:%d-%b} (dB)")
axs[1].imshow(real_map[r0:r1, c0:c1], cmap="Blues", vmin=0, vmax=1, interpolation="nearest"); axs[1].set_title("Etiquetado: inundación nueva")
im = axs[2].imshow(prob_map[r0:r1, c0:c1], cmap="magma_r", vmin=0, vmax=1, interpolation="nearest")
axs[2].set_title(f"Modelo B ({MEJOR_B}) · {eval_mapa.iloc[0]['captura_top_5%']:.0%} de lo nuevo en el 5 % más probable", fontsize=9)
plt.colorbar(im, ax=axs[2], fraction=0.04, label="probabilidad")
for a in axs: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.savefig(DIR_OUT / "B_mapa_E6_A_T148.png"); plt.show()
del cache

joblib.dump({"modelo": mod_b, "umbral": u_b, "variables": VAR_B, "nombre": MEJOR_B, "malla": {"x0": X0, "y1": Y1, "res": RES, "forma": [NY, NX]},
             "entrenado_con": "pares con t+1 < 2026-08-20", "metricas_E6": ficha["metricas_prueba_E6"]},
            DIR_OUT / "modeloB_espacial.joblib")
print("Escrito:", nombre_tif, "+ ficha .json · SHA-256", sha[:16], "…")

Celdas evaluadas: 6,767,000 · inundación nueva (etiquetado): 13.3 km² (0.12%) · zona de alerta (5 % más probable): 541 km² · umbral 0.02
                            puntaje  PR_AUC  ROC_AUC  captura_top_1%  captura_top_5%  captura_top_10%
0                           XGBoost   0.022    0.904           0.231           0.422            0.612
1  Persistencia (inund. nueva en t)   0.002    0.504           0.009           0.047            0.205
2  Regla de terreno (HAND y d_agua)   0.005    0.875           0.006           0.313            0.463


Escrito: CHOCO_A_T148_E6_prob_inundacion_20260904.tif + ficha .json · SHA-256 1f8bba3128149ae1 …


## B5b. Scenarios for the app traffic light (low · medium · high)

In [17]:
ESCENARIOS = {"bajo": float(q.loc[ENTRENA[0]:ENTRENA[1]].quantile(0.50)),
              "medio": UMBRAL_NIVEL,
              "alto": float(q.loc[ENTRENA[0]:ENTRENA[1]].quantile(0.99))}
filas_esc = []
for nombre_esc, nivel in ESCENARIOS.items():
    X = todo[VAR_B].copy()
    X["nivel_t1"] = nivel
    X["nivel_max_tramo"] = np.maximum(X["nivel_t"], nivel)
    X["dnivel"] = nivel - X["nivel_t"]
    X["nivel_max_rel_P90"] = X["nivel_max_tramo"] / UMBRAL_NIVEL
    pe = np.concatenate([mod_b.predict_proba(X.iloc[i:i + 500_000])[:, 1] for i in range(0, len(X), 500_000)])
    m_esc = np.full((NY, NX), 255, np.uint8)
    m_esc[todo.fila.values, todo.col.values] = np.clip(np.round(pe * 100), 0, 100).astype(np.uint8)
    nombre_e = f"CHOCO_{STACK_DEMO}_escenario_{nombre_esc}.tif"
    with rasterio.open(DIR_OUT / nombre_e, "w", **perfil) as dst:
        dst.write(m_esc, 1)
        dst.update_tags(TIPO_PRODUCTO="probabilidad_inundacion_nisar", UNIDADES="porcentaje", ESCENARIO=nombre_esc)
    ficha_e = json.loads(json.dumps(ficha))
    ficha_e["producto"] = f"Probabilidad de inundación nueva · escenario {nombre_esc} (nivel en Quibdó {nivel:.0f} cm)"
    ficha_e["escenario"] = {"nombre": nombre_esc, "nivel_quibdo_cm": round(nivel), "estado_base": f"{T_DEMO:%Y-%m-%d}"}
    ficha_e["fechas_detalle"]["fecha_objetivo"] = None
    ficha_e["fechas_detalle"]["nota"] = "Escenario hipotético: mismo estado del radar, nivel del río fijado por el usuario o por el Modelo A."
    ficha_e.pop("metricas_prueba_E6", None)
    ficha_e["procedencia"]["archivo"] = {"nombre": nombre_e, "sha256": hashlib.sha256((DIR_OUT / nombre_e).read_bytes()).hexdigest()}
    (DIR_OUT / nombre_e.replace(".tif", ".json")).write_text(json.dumps(ficha_e, ensure_ascii=False, indent=2), encoding="utf-8")
    alerta = pe >= u_b
    filas_esc.append({"escenario": nombre_esc, "nivel_cm": round(nivel), "km2_sobre_umbral": round(alerta.sum() * RES**2 / 1e6, 1),
                      "prob_media_%": round(pe.mean() * 100, 2)})
pd.DataFrame(filas_esc)

,escenario,nivel_cm,km2_sobre_umbral,prob_media_%
0,bajo,338,597.8,1.29
1,medio,497,541.4,1.03
2,alto,588,541.4,1.03


## B6. Summary for the report

In [18]:
resumen = {
    "modelo_A": {k: {"modelo": v["nombre"], "conjunto": v["conjunto"], "H_dias": v["H_dias"],
                     "umbral_nivel_cm": round(v["umbral_nivel_cm"]),
                     **{m: round(float(x), 3) for m, x in v["metricas_prueba"].items()}} for k, v in ELEGIDOS.items()},
    "modelo_B": {"modelo": MEJOR_B, "umbral": round(float(u_b), 3), "E6_A_T148": ficha["metricas_prueba_E6"],
                 "filas_entrenamiento": int(len(datos_B)), "pares": len(PARES)},
}
(DIR_OUT / "resumen_resultados.json").write_text(json.dumps(resumen, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(resumen, ensure_ascii=False, indent=2))

{
  "modelo_A": {
    "operativo": {
      "modelo": "Random Forest",
      "conjunto": "2 · + GloFAS hoy",
      "H_dias": 3,
      "umbral_nivel_cm": 497,
      "ROC_AUC": 0.767,
      "PR_AUC": 0.443,
      "F1": 0.492,
      "POD": 0.519,
      "FAR": 0.533,
      "CSI": 0.326,
      "Brier": 0.118,
      "umbral": 0.333
    },
    "con_pronostico": {
      "modelo": "XGBoost",
      "conjunto": "3 · + pronóstico GloFAS (techo)",
      "H_dias": 3,
      "umbral_nivel_cm": 497,
      "ROC_AUC": 0.79,
      "PR_AUC": 0.458,
      "F1": 0.477,
      "POD": 0.67,
      "FAR": 0.629,
      "CSI": 0.314,
      "Brier": 0.118,
      "umbral": 0.207
    },
    "solo_ideam": {
      "modelo": "Random Forest",
      "conjunto": "1 · Quibdó IDEAM + ONI",
      "H_dias": 3,
      "umbral_nivel_cm": 497,
      "ROC_AUC": 0.766,
      "PR_AUC": 0.45,
      "F1": 0.485,
      "POD": 0.558,
      "FAR": 0.57,
      "CSI": 0.321,
      "Brier": 0.118,
      "umbral": 0.331
    }
  },
  "modelo_B":